# SROIE Benchmark — InternVL3.5-8B (vLLM Data-Parallel)

**ICDAR 2019 SROIE Task 3: Receipt Key Information Extraction**

This notebook benchmarks [InternVL3.5-8B](https://huggingface.co/OpenGVLab/InternVL3-8B) on the
SROIE dataset using **vLLM** with **data parallelism** — one independent engine per GPU,
images partitioned round-robin across workers. Scales linearly with GPU count.

| Field | Description |
|-------|-------------|
| **company** | Store / business name |
| **date** | Receipt date (DD/MM/YYYY) |
| **address** | Full store address |
| **total** | Final total amount |

**Metrics:** Per-field precision, recall, and F1 (entity-level exact match after normalization),
plus overall F1 (mean of per-field F1).

**Self-contained:** All evaluation, preprocessing, and inference logic is inlined.
Requires `vllm`, `Pillow`, and `tqdm`.

---
## 1. Configuration

In [1]:
from pathlib import Path

# --- Paths ---
DATA_DIR = Path("data/sroie")  # Created automatically by the download cell below
MODEL_PATH = Path("/home/jovyan/nfs_share/models/InternVL3_5-8B")

# --- Data-parallel settings ---
NUM_GPUS = 2  # Number of DP workers (one vLLM engine per GPU)
GPU_MEMORY_UTILIZATION = 0.90
MAX_MODEL_LEN = 8192

# --- Inference settings ---
MAX_IMAGES = None  # Set to an int (e.g. 10) for quick test runs
MAX_NEW_TOKENS = 256

---
## 2. Dataset Setup

The SROIE dataset is downloaded automatically from Kaggle into `data/sroie/` (relative to
this notebook) if not already present. The `kaggle` CLI is installed automatically if needed.
Requires valid Kaggle API credentials at `~/.kaggle/kaggle.json`.

```
data/sroie/
  test/
    img/          # Receipt images (.jpg)
    entities/     # Ground truth files (.txt with JSON content)
  train/
    img/
    entities/
```

Each ground-truth file is a JSON object with keys: `company`, `date`, `address`, `total`.

In [2]:
import shutil
import subprocess
import sys
import zipfile

DATA_DIR.mkdir(parents=True, exist_ok=True)

img_dir = DATA_DIR / "test" / "img"
key_dir = DATA_DIR / "test" / "entities"

if not img_dir.exists() or not key_dir.exists():
    # Resolve kaggle CLI: prefer PATH, fall back to environment bin/ after pip install
    kaggle_cmd = shutil.which("kaggle")
    if kaggle_cmd is None:
        print("Installing kaggle CLI ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
        kaggle_cmd = str(Path(sys.executable).parent / "kaggle")

    print("SROIE dataset not found — downloading from Kaggle ...")
    zip_path = DATA_DIR / "sroie-datasetv2.zip"
    subprocess.run(
        [kaggle_cmd, "datasets", "download", "-d", "urbikn/sroie-datasetv2", "-p", str(DATA_DIR)],
        check=True,
    )
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_DIR)
    zip_path.unlink()
    # Kaggle zip nests everything under SROIE2019/ — move contents up
    nested = DATA_DIR / "SROIE2019"
    if nested.exists():
        for child in nested.iterdir():
            dest = DATA_DIR / child.name
            if not dest.exists():
                child.rename(dest)
        shutil.rmtree(nested)
    print("Download complete.")

assert img_dir.exists(), f"Image directory not found: {img_dir}"
assert key_dir.exists(), f"Key directory not found: {key_dir}"
print(f"Images: {len(list(img_dir.iterdir()))} files")
print(f"Keys:   {len(list(key_dir.iterdir()))} files")

SROIE dataset not found — downloading from Kaggle ...
Dataset URL: https://www.kaggle.com/datasets/urbikn/sroie-datasetv2
License(s): other


100%|██████████| 834M/834M [00:37<00:00, 23.1MB/s] 



Download complete.
Images: 347 files
Keys:   347 files


---
## 3. Imports

In [ ]:
import gc
import json
import re
import subprocess
import sys
import time
from dataclasses import dataclass, field

from PIL import Image
from tqdm.auto import tqdm

---
## 4. Ground Truth Loading

In [ ]:
SROIE_FIELDS = ("company", "date", "address", "total")


def load_sroie_ground_truth(key_dir: Path) -> dict[str, dict[str, str]]:
    """Load per-image JSON ground truth from SROIE key/ directory.

    Each file contains a JSON object with keys: company, date, address, total.
    Falls back to line-separated format if JSON parsing fails.
    """
    ground_truth: dict[str, dict[str, str]] = {}

    for txt_path in sorted([*key_dir.glob("*.json"), *key_dir.glob("*.txt")]):
        image_id = txt_path.stem
        text = txt_path.read_text(encoding="utf-8").strip()
        if not text:
            continue

        try:
            data = json.loads(text)
        except json.JSONDecodeError:
            # Some SROIE files have line-separated values instead of JSON
            lines = text.splitlines()
            if len(lines) >= 4:
                data = {
                    "company": lines[0].strip(),
                    "date": lines[1].strip(),
                    "address": lines[2].strip(),
                    "total": lines[3].strip(),
                }
            else:
                continue

        gt_entry = {}
        for f in SROIE_FIELDS:
            gt_entry[f] = data.get(f, "")
        ground_truth[image_id] = gt_entry

    return ground_truth


ground_truth = load_sroie_ground_truth(key_dir)
print(f"Loaded {len(ground_truth)} ground truth entries")

---
## 5. Evaluation Functions

In [ ]:
# --- Result dataclasses ---


@dataclass
class SROIEFieldResult:
    """Aggregate metrics for one SROIE field across all images."""

    field: str
    true_positives: int = 0
    false_positives: int = 0
    false_negatives: int = 0

    @property
    def precision(self) -> float:
        denom = self.true_positives + self.false_positives
        return self.true_positives / denom if denom > 0 else 0.0

    @property
    def recall(self) -> float:
        denom = self.true_positives + self.false_negatives
        return self.true_positives / denom if denom > 0 else 0.0

    @property
    def f1(self) -> float:
        p, r = self.precision, self.recall
        return 2 * p * r / (p + r) if (p + r) > 0 else 0.0


@dataclass
class SROIEImageResult:
    """Per-image extraction result."""

    image_id: str
    ground_truth: dict[str, str]
    predicted: dict[str, str]
    matches: dict[str, bool] = field(default_factory=dict)


@dataclass
class SROIEBenchmarkResult:
    """Full benchmark result for one model run."""

    model_name: str
    image_results: list[SROIEImageResult]
    field_results: dict[str, SROIEFieldResult]
    overall_f1: float
    total_images: int
    elapsed_seconds: float = 0.0


# --- Normalization ---

_WHITESPACE_RE = re.compile(r"\s+")
_CURRENCY_RE = re.compile(r"[$\u00a3\u20ac\u00a5RM]")
_COMMA_IN_NUMBER_RE = re.compile(r"(\d),(\d)")


def normalize_text(text: str) -> str:
    """General text normalization: lowercase, collapse whitespace, strip."""
    text = text.lower().strip()
    text = _WHITESPACE_RE.sub(" ", text)
    return text


def normalize_total(text: str) -> str:
    """Normalize monetary total: strip currency symbols, commas, whitespace."""
    text = text.strip()
    text = _CURRENCY_RE.sub("", text)
    text = _COMMA_IN_NUMBER_RE.sub(r"\1\2", text)
    text = text.strip()
    try:
        val = float(text)
        return f"{val:.2f}"
    except ValueError:
        return normalize_text(text)


def normalize_date(text: str) -> str:
    """Normalize date to DD/MM/YYYY format."""
    text = text.strip()

    # ISO format: YYYY-MM-DD
    m = re.match(r"(\d{4})[-/.](\d{1,2})[-/.](\d{1,2})", text)
    if m:
        return f"{int(m.group(3)):02d}/{int(m.group(2)):02d}/{m.group(1)}"

    # DD/MM/YYYY or DD-MM-YYYY or DD.MM.YYYY
    m = re.match(r"(\d{1,2})[-/.](\d{1,2})[-/.](\d{4})", text)
    if m:
        return f"{int(m.group(1)):02d}/{int(m.group(2)):02d}/{m.group(3)}"

    # DD Mon YYYY
    months = {
        "jan": "01",
        "feb": "02",
        "mar": "03",
        "apr": "04",
        "may": "05",
        "jun": "06",
        "jul": "07",
        "aug": "08",
        "sep": "09",
        "oct": "10",
        "nov": "11",
        "dec": "12",
    }
    m = re.match(r"(\d{1,2})\s+(\w{3,})\s+(\d{4})", text, re.IGNORECASE)
    if m:
        month_str = m.group(2)[:3].lower()
        if month_str in months:
            return f"{int(m.group(1)):02d}/{months[month_str]}/{m.group(3)}"

    return normalize_text(text)


# --- Matching ---


def sroie_field_match(field_name: str, predicted: str, ground_truth: str) -> bool:
    """Check exact match after field-specific normalization."""
    if not predicted or not ground_truth:
        return False

    if field_name == "total":
        return normalize_total(predicted) == normalize_total(ground_truth)
    if field_name == "date":
        return normalize_date(predicted) == normalize_date(ground_truth)
    return normalize_text(predicted) == normalize_text(ground_truth)


# --- Aggregation ---


def compute_sroie_metrics(
    image_results: list[SROIEImageResult],
    model_name: str = "unknown",
    elapsed_seconds: float = 0.0,
) -> SROIEBenchmarkResult:
    """Aggregate per-image results into per-field and overall metrics."""
    field_results = {f: SROIEFieldResult(field=f) for f in SROIE_FIELDS}

    for img_result in image_results:
        for f in SROIE_FIELDS:
            gt_val = img_result.ground_truth.get(f, "")
            pred_val = img_result.predicted.get(f, "")

            match = sroie_field_match(f, pred_val, gt_val)
            img_result.matches[f] = match

            if match:
                field_results[f].true_positives += 1
            else:
                if pred_val:
                    field_results[f].false_positives += 1
                if gt_val:
                    field_results[f].false_negatives += 1

    f1_scores = [fr.f1 for fr in field_results.values()]
    overall_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0.0

    return SROIEBenchmarkResult(
        model_name=model_name,
        image_results=image_results,
        field_results=field_results,
        overall_f1=overall_f1,
        total_images=len(image_results),
        elapsed_seconds=elapsed_seconds,
    )


print("Evaluation functions loaded.")

---
## 6. Data-Parallel Worker

Each GPU runs an independent vLLM engine (TP=1) in its own subprocess, pinned via
`CUDA_VISIBLE_DEVICES`. Images are partitioned round-robin across workers for load
balance, then results are merged back in original order.

The cell below writes a self-contained worker script to disk.

In [ ]:
_WORKER_SCRIPT = DATA_DIR / "_sroie_dp_worker.py"

_WORKER_SCRIPT.write_text(r'''#!/usr/bin/env python3
"""SROIE benchmark DP worker — processes a shard of images on one GPU."""
import argparse
import base64
import io
import json
import os
import sys

from PIL import Image


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--gpu-id", type=int, required=True)
    parser.add_argument("--model-path", required=True)
    parser.add_argument("--image-list", required=True)
    parser.add_argument("--prompt-file", required=True)
    parser.add_argument("--output-file", required=True)
    parser.add_argument("--max-tokens", type=int, default=256)
    parser.add_argument("--max-model-len", type=int, default=8192)
    parser.add_argument("--gpu-mem", type=float, default=0.90)
    args = parser.parse_args()

    os.environ["CUDA_VISIBLE_DEVICES"] = str(args.gpu_id)
    os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")

    from vllm import LLM, SamplingParams

    print(f"[GPU {args.gpu_id}] Loading model ...", flush=True)
    model = LLM(
        model=args.model_path,
        tensor_parallel_size=1,
        max_model_len=args.max_model_len,
        gpu_memory_utilization=args.gpu_mem,
        max_num_seqs=1,
        limit_mm_per_prompt={"image": 1},
        trust_remote_code=True,
        disable_log_stats=True,
        enforce_eager=False,
        enable_prefix_caching=True,
    )

    sampling = SamplingParams(max_tokens=args.max_tokens, temperature=0)

    with open(args.image_list) as f:
        image_paths = json.load(f)
    with open(args.prompt_file) as f:
        prompt = f.read()

    results = []
    total = len(image_paths)
    print(f"[GPU {args.gpu_id}] Processing {total} images ...", flush=True)

    for idx, img_path in enumerate(image_paths):
        try:
            image = Image.open(img_path).convert("RGB")
            buf = io.BytesIO()
            image.save(buf, format="PNG")
            data_uri = (
                f"data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}"
            )
            buf.close()

            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image_url", "image_url": {"url": data_uri}},
                        {"type": "text", "text": prompt},
                    ],
                }
            ]

            outputs = model.chat(
                messages=messages, sampling_params=sampling, use_tqdm=False
            )
            text = outputs[0].outputs[0].text.strip()
            del outputs, messages, data_uri, image
            results.append({"image_path": img_path, "response": text})
        except Exception as e:
            results.append({"image_path": img_path, "response": "", "error": str(e)})

        if (idx + 1) % 50 == 0 or idx + 1 == total:
            print(f"[GPU {args.gpu_id}] {idx + 1}/{total}", flush=True)

    with open(args.output_file, "w") as f:
        json.dump(results, f)

    del model
    print(f"[GPU {args.gpu_id}] Done.", flush=True)


if __name__ == "__main__":
    main()
''')
print(f"Worker script: {_WORKER_SCRIPT}")

---
## 7. Response Parsing

In [ ]:
_FIELD_RE = re.compile(
    r"^(company|date|address|total)\s*:\s*(.+)$",
    re.IGNORECASE | re.MULTILINE,
)


def parse_sroie_response(raw_response: str) -> dict[str, str]:
    """Parse model response into SROIE field dict.

    Expects lines like:
        company: SOME STORE NAME
        date: 25/12/2018
        address: 123 Main St
        total: 4.95
    """
    result: dict[str, str] = {}
    for match in _FIELD_RE.finditer(raw_response):
        field_name = match.group(1).lower()
        value = match.group(2).strip()
        if value.upper() != "NOT_FOUND":
            result[field_name] = value
    return result

---
## 8. Extraction Prompt

In [ ]:
SROIE_PROMPT = """Extract the following 4 fields from this receipt image.
Return each field on its own line in exactly this format:

company: <the store or company name>
date: <the receipt date in DD/MM/YYYY format>
address: <the full store address>
total: <the total amount as a number, no currency symbol, e.g. 4.95>

Rules:
- For "company", use the business name printed at the top of the receipt.
- For "date", convert to DD/MM/YYYY format regardless of how it appears.
- For "address", include the full street address as printed.
- For "total", use only the final total amount. Remove any currency symbol.
- If a field is not visible, write NOT_FOUND.
- Do NOT add any extra text, explanations, or formatting.
"""

print(f"Prompt length: {len(SROIE_PROMPT)} chars")

---
## 9. Benchmark Loop

In [ ]:
# Discover images with matching ground truth
image_extensions = {".jpg", ".jpeg", ".png"}
all_images = sorted(
    p for p in img_dir.iterdir() if p.suffix.lower() in image_extensions and p.stem in ground_truth
)

if MAX_IMAGES:
    all_images = all_images[:MAX_IMAGES]

# Partition images across GPUs (round-robin for load balance)
shards: list[list[str]] = [[] for _ in range(NUM_GPUS)]
for idx, img_path in enumerate(all_images):
    shards[idx % NUM_GPUS].append(str(img_path))

# Write prompt and shard manifests
prompt_file = DATA_DIR / "_prompt.txt"
prompt_file.write_text(SROIE_PROMPT)

print(f"Running benchmark: {len(all_images)} images across {NUM_GPUS} GPUs")
for i, shard in enumerate(shards):
    print(f"  GPU {i}: {len(shard)} images")

# Build worker environment: prepend conda's lib/ so the subprocess Python
# finds the conda libstdc++ instead of the older system copy at /usr/lib64.
import os
import selectors

worker_env = os.environ.copy()
conda_prefix = worker_env.get("CONDA_PREFIX", "")
if conda_prefix:
    conda_lib = os.path.join(conda_prefix, "lib")
    ld_path = worker_env.get("LD_LIBRARY_PATH", "")
    if conda_lib not in ld_path:
        worker_env["LD_LIBRARY_PATH"] = conda_lib + (":" + ld_path if ld_path else "")
        print(f"  LD_LIBRARY_PATH = {worker_env['LD_LIBRARY_PATH']}")

# Launch DP workers
start_time = time.time()
procs = []
output_files = []

for gpu_id in range(NUM_GPUS):
    shard_file = DATA_DIR / f"_shard_{gpu_id}.json"
    output_file = DATA_DIR / f"_results_{gpu_id}.json"
    shard_file.write_text(json.dumps(shards[gpu_id]))
    output_files.append(output_file)

    cmd = [
        sys.executable,
        str(_WORKER_SCRIPT),
        "--gpu-id",
        str(gpu_id),
        "--model-path",
        str(MODEL_PATH),
        "--image-list",
        str(shard_file),
        "--prompt-file",
        str(prompt_file),
        "--output-file",
        str(output_file),
        "--max-tokens",
        str(MAX_NEW_TOKENS),
        "--max-model-len",
        str(MAX_MODEL_LEN),
        "--gpu-mem",
        str(GPU_MEMORY_UTILIZATION),
    ]
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=worker_env,
    )
    procs.append(proc)

# Stream worker output in real time from the main thread (Jupyter requirement).
# Uses selectors to multiplex reads from all worker pipes concurrently.
sel = selectors.DefaultSelector()
for gpu_id, proc in enumerate(procs):
    sel.register(proc.stdout, selectors.EVENT_READ, gpu_id)

active = len(procs)
while active > 0:
    for key, _ in sel.select(timeout=1.0):
        line = key.fileobj.readline()
        if line:
            print(line, end="", flush=True)
        else:
            sel.unregister(key.fileobj)
            active -= 1

for gpu_id, proc in enumerate(procs):
    proc.wait()
    if proc.returncode != 0:
        print(f"ERROR: Worker GPU {gpu_id} failed (exit {proc.returncode})")

elapsed = time.time() - start_time

# Merge results from all workers
response_map: dict[str, str] = {}
for output_file in output_files:
    with output_file.open() as f:
        for r in json.load(f):
            response_map[r["image_path"]] = r["response"]

# Build image_results in original order
image_results: list[SROIEImageResult] = []
for img_path in all_images:
    image_id = img_path.stem
    gt = ground_truth[image_id]
    raw_response = response_map.get(str(img_path), "")
    predicted = parse_sroie_response(raw_response)
    image_results.append(SROIEImageResult(image_id=image_id, ground_truth=gt, predicted=predicted))

# Cleanup temp files
for gpu_id in range(NUM_GPUS):
    (DATA_DIR / f"_shard_{gpu_id}.json").unlink(missing_ok=True)
    (DATA_DIR / f"_results_{gpu_id}.json").unlink(missing_ok=True)
prompt_file.unlink(missing_ok=True)

print(
    f"\nDone. {len(image_results)} images in {elapsed:.1f}s "
    f"({len(image_results) / elapsed * 60:.1f} images/min)"
)

---
## 10. Results

In [ ]:
result = compute_sroie_metrics(image_results, "InternVL3.5-8B", elapsed)

# --- Per-field table ---
print(f"{'Field':<12} {'Precision':>10} {'Recall':>10} {'F1':>10}  {'TP':>5} {'FP':>5} {'FN':>5}")
print("-" * 65)
for f in SROIE_FIELDS:
    fr = result.field_results[f]
    print(
        f"{f:<12} {fr.precision * 100:>9.1f}% {fr.recall * 100:>9.1f}% {fr.f1 * 100:>9.1f}%"
        f"  {fr.true_positives:>5} {fr.false_positives:>5} {fr.false_negatives:>5}"
    )
print("-" * 65)
print(f"{'Overall F1':<12} {'':>10} {'':>10} {result.overall_f1 * 100:>9.1f}%")

# --- Throughput ---
print(f"\nImages: {result.total_images}")
print(f"Time:   {result.elapsed_seconds:.1f}s")
if result.elapsed_seconds > 0:
    print(f"Speed:  {result.total_images / result.elapsed_seconds * 60:.1f} images/min")

---
## 11. Error Analysis

In [ ]:
# Show mismatched predictions for debugging
errors = [img for img in result.image_results if not all(img.matches.get(f, False) for f in SROIE_FIELDS)]

print(f"{len(errors)} images with at least one field mismatch:\n")

for img in errors[:20]:  # Show first 20
    mismatched = [f for f in SROIE_FIELDS if not img.matches.get(f, False)]
    print(f"--- {img.image_id} (mismatched: {', '.join(mismatched)}) ---")
    for f in mismatched:
        gt_val = img.ground_truth.get(f, "")
        pred_val = img.predicted.get(f, "<missing>")
        print(f"  {f:<10} GT:   {gt_val}")
        print(f"  {'':<10} PRED: {pred_val}")
    print()

---
## 12. Export Results (Optional)

In [ ]:
import csv

output_dir = DATA_DIR / "output"
output_dir.mkdir(parents=True, exist_ok=True)

# Per-image CSV
csv_path = output_dir / "sroie_internvl3_per_image.csv"
with csv_path.open("w", newline="") as f:
    writer = csv.writer(f)
    header = ["image_id"]
    for field_name in SROIE_FIELDS:
        header.extend([f"{field_name}_gt", f"{field_name}_pred", f"{field_name}_match"])
    writer.writerow(header)

    for img in result.image_results:
        row = [img.image_id]
        for field_name in SROIE_FIELDS:
            row.extend(
                [
                    img.ground_truth.get(field_name, ""),
                    img.predicted.get(field_name, ""),
                    img.matches.get(field_name, False),
                ]
            )
        writer.writerow(row)

print(f"Per-image results saved to {csv_path}")

# Summary JSON
summary = {
    "model": result.model_name,
    "overall_f1": round(result.overall_f1, 4),
    "total_images": result.total_images,
    "elapsed_seconds": round(result.elapsed_seconds, 2),
    "per_field": {
        f: {
            "precision": round(result.field_results[f].precision, 4),
            "recall": round(result.field_results[f].recall, 4),
            "f1": round(result.field_results[f].f1, 4),
        }
        for f in SROIE_FIELDS
    },
}
json_path = output_dir / "sroie_internvl3_summary.json"
json_path.write_text(json.dumps(summary, indent=2))
print(f"Summary saved to {json_path}")

---
## 13. Cleanup

In [ ]:
# Remove the worker script written to disk
if _WORKER_SCRIPT.exists():
    _WORKER_SCRIPT.unlink()
    print(f"Removed {_WORKER_SCRIPT}")

gc.collect()
print("Cleanup complete.")